# Healthcare accessibility in Belarus

This notebook combines medical facilities, settlements, administrative boundaries, population statistics, and salary statistics.

## Project folders

- `map_data/` — original geographic source files;
- `belstat/` — original Belstat CSV files;
- `output_data/` — files created by the notebooks and QGIS.

It produces:

- straight-line distance from every settlement to its nearest medical facility;
- regional and district accessibility statistics;
- district and regional demographic and salary datasets;
- population estimates for settlements located more than one hour from medical care;
- GeoJSON files used in the Mapbox visualization.

Walking-time service areas were calculated separately in QGIS. This notebook uses the exported list of settlements outside the one-hour service areas.

## 1. Imports

All imports are kept in one place so the notebook can be run from top to bottom.

In [1]:
import pandas as pd
import geopandas as gpd

from rapidfuzz import process, fuzz
from shapely.geometry import LineString

## 2. Load and prepare medical facilities

The facility dataset was scraped from Healthcare.by and geocoded earlier.

Before creating geometries, the closed Velikodoletskaya nursing-care hospital is removed and the missing hospital in Kholomerye is added manually. Coordinates are converted to numeric values and rows without valid coordinates are dropped.

In [2]:
healthcare_df = pd.read_csv("output_data/healthcare_regions.csv")

# Remove the closed facility from the current-access analysis (OPTIONAL)
# closed_facility = "Великодолецкая больница сестринского ухода"

# closed_mask = (
#     healthcare_df["facility_title"]
#     .astype("string")
#     .str.strip()
#     .eq(closed_facility)
# )

# print("Closed facilities removed:", closed_mask.sum())

# healthcare_df = (
#     healthcare_df.loc[~closed_mask]
#     .copy()
#     .reset_index(drop=True)
# )

In [3]:
# Add the missing hospital only if it is not already present
new_facility = pd.DataFrame(
    [
        {
            "facility_title": "Холомерская сельская участковая больница",
            "link": pd.NA,
            "settlement": "Холомерье",
            "facility_type": "Больница",
            "facility_type_eng": "Hospital",
            "address": (
                "ул. Советская, 2А, 211581, д. Холомерье, "
                "Городокский р-н, Витебская обл."
            ),
            "website": pd.NA,
            "latitude": 55.743639,
            "longitude": 29.675528,
            "region": "Vitebsk",
            "district": "Gorodok",
        }
    ]
)

if not healthcare_df["facility_title"].eq(
    "Холомерская сельская участковая больница"
).any():
    healthcare_df = pd.concat(
        [healthcare_df, new_facility],
        ignore_index=True,
    )

# Convert coordinates to numbers
healthcare_df["latitude"] = pd.to_numeric(
    healthcare_df["latitude"],
    errors="coerce",
)

healthcare_df["longitude"] = pd.to_numeric(
    healthcare_df["longitude"],
    errors="coerce",
)

healthcare_df = healthcare_df.dropna(
    subset=["latitude", "longitude"]
).copy()

# Create point geometry: X is longitude and Y is latitude
healthcare = gpd.GeoDataFrame(
    healthcare_df,
    geometry=gpd.points_from_xy(
        healthcare_df["longitude"],
        healthcare_df["latitude"],
    ),
    crs="EPSG:4326",
)

print("Medical facilities retained:", len(healthcare))

Medical facilities retained: 3475


## 3. Load settlements and add population

`final_settlements.gpkg` contains the cleaned settlement points prepared in the previous notebook.

Administrative columns are renamed immediately to `region` and `district`. This avoids renaming them halfway through the analysis and prevents later references to a mixture of old and new column names.

Population comes from the original HOTOSM populated-places file. It is cleaned once here and then carried through the rest of the notebook.

In [4]:
settlements = gpd.read_file(
    "output_data/final_settlements.gpkg"
).rename(
    columns={
        "adm1_name": "region",
        "adm2_name": "district",
    }
)

settlements = settlements.to_crs("EPSG:4326")

settlements_raw = gpd.read_file(
    "map_data/populated_places.gpkg"
)

# Keep one population value for each settlement ID
population_lookup = (
    settlements_raw.loc[
        settlements_raw.geometry.geom_type.eq("Point"),
        ["id", "population"],
    ]
    .drop_duplicates(subset="id")
    .copy()
)


def clean_population(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(r"[^0-9]", "", regex=True),
        errors="coerce",
    )


population_lookup["population_num"] = clean_population(
    population_lookup["population"]
)

settlements = settlements.merge(
    population_lookup,
    on="id",
    how="left",
)

print("Settlements loaded:", len(settlements))
print(
    "Settlements with known population:",
    settlements["population_num"].notna().sum(),
)

Settlements loaded: 22477
Settlements with known population: 5303


## 4. Find the nearest medical facility

Straight-line distances must be calculated in metres, not geographic degrees. Both datasets are therefore projected to an automatically selected UTM coordinate reference system.

`gpd.sjoin_nearest()` assigns the nearest medical facility to every settlement and records the distance in metres.

In [5]:
# Select a projected CRS suitable for the dataset extent
metric_crs = settlements.estimate_utm_crs()

settlements_m = settlements.to_crs(metric_crs)
healthcare_m = healthcare.to_crs(metric_crs)

print("Metric CRS:", metric_crs)

access = gpd.sjoin_nearest(
    settlements_m,
    healthcare_m[
        [
            "facility_title",
            "facility_type",
            "facility_type_eng",
            "link",
            "address",
            "geometry",
        ]
    ],
    how="left",
    distance_col="distance_m",
)

access = access.drop(
    columns="index_right",
    errors="ignore",
)

access = access.rename(
    columns={
        "facility_title": "nearest_facility",
        "facility_type": "nearest_facility_type",
        "facility_type_eng": "nearest_facility_type_eng",
        "link": "nearest_facility_link",
        "address": "nearest_facility_address",
    }
)

access["distance_km"] = access["distance_m"] / 1000
access["near_facility"] = access["distance_m"] <= 5000

# Return point geometry to longitude and latitude
access = access.to_crs("EPSG:4326")

Metric CRS: EPSG:32635


## 5. Remove records outside the analysis area

One known erroneous settlement record in the Stolin district is removed explicitly.

The administrative code is then used to keep Belarusian settlements only. A separate boundary check reports how many points fall outside the GADM country polygon, but it does not silently remove additional border settlements.

In [6]:
# Remove one known erroneous settlement record
zabolotye_mask = (
    access["name_ru"].eq("Заболотье")
    & access["region"].eq("Brest")
    & access["district"].eq("Stolin")
)

print("Known erroneous records removed:", zabolotye_mask.sum())

access = (
    access.loc[~zabolotye_mask]
    .copy()
    .reset_index(drop=True)
)

# Keep settlements with a Belarusian ADM1 code
belarus_code_mask = (
    access["adm1_pcode"]
    .astype("string")
    .str.startswith("BY", na=False)
)

access_clean = (
    access.loc[belarus_code_mask]
    .copy()
    .reset_index(drop=True)
)

print("Records removed by administrative code:", (~belarus_code_mask).sum())
print("Settlements retained:", len(access_clean))

Known erroneous records removed: 1
Records removed by administrative code: 24
Settlements retained: 23816


In [7]:
# Geographic quality check against the Belarus country boundary
belarus = gpd.read_file(
    "map_data/administrative_boundaries_belarus_gadm41.gpkg",
    layer="ADM_ADM_0",
).to_crs(access_clean.crs)

belarus_geometry = belarus.geometry.union_all()

inside_belarus = (
    access_clean.geometry.notna()
    & ~access_clean.geometry.is_empty
    & access_clean.geometry.intersects(belarus_geometry)
)

print(
    "Retained settlements outside the GADM polygon:",
    (~inside_belarus).sum(),
)

Retained settlements outside the GADM polygon: 65


## 6. Validate and save the nearest-facility results

The checks below confirm that the spatial join produced distances and facility information, and that settlement rows are not duplicated by their index.

Only the cleaned result is exported, avoiding an unnecessary intermediate copy.

In [8]:
print("Missing distances:", access_clean["distance_m"].isna().sum())
print("Missing nearest facilities:", access_clean["nearest_facility"].isna().sum())
print(
    "Missing facility types:",
    access_clean["nearest_facility_type"].isna().sum(),
)
print("Duplicated indexes:", access_clean.index.duplicated().sum())

access_clean.to_file(
    "output_data/settlements_nearest_healthcare_clean.gpkg",
    layer="settlement_access",
    driver="GPKG",
    index=False,
)

access_clean.drop(columns="geometry").to_csv(
    "output_data/settlements_nearest_healthcare_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

Missing distances: 0
Missing nearest facilities: 0
Missing facility types: 0
Duplicated indexes: 0


## 7. Overall distance statistics

The analysis uses three straight-line distance thresholds: 3 km, 8 km, and 13 km. Counts and percentages are calculated from the same cleaned settlement table.

In [9]:
thresholds_km = [3, 8, 13]

for distance_km in thresholds_km:
    farther = access_clean["distance_m"] > distance_km * 1000

    print(
        f"More than {distance_km} km: "
        f"{farther.sum():,} settlements "
        f"({farther.mean() * 100:.1f}%)"
    )

More than 3 km: 13,361 settlements (56.1%)
More than 8 km: 924 settlements (3.9%)
More than 13 km: 19 settlements (0.1%)


In [10]:
# Distance distribution by settlement type
distance_by_place = (
    access_clean.groupby("place")["distance_km"]
    .describe()
    .round(2)
)

distance_by_place

,count,mean,std,min,25%,50%,75%,max
place,,,,,,,,
city,19.0,0.37,0.31,0.03,0.11,0.24,0.57,1.11
hamlet,20361.0,3.77,2.13,0.00,2.29,3.48,4.96,19.17
isolated_dwelling,674.0,3.94,2.14,0.07,2.32,3.61,5.00,12.05
town,195.0,0.73,0.73,0.02,0.28,0.50,0.92,5.96
village,2567.0,1.93,2.33,0.00,0.24,0.65,3.33,12.30


## 8. Regional and district statistics

The same function is used for both administrative levels. This prevents the regional and district calculations from drifting apart or using different units.

All distance summaries are stored in kilometres.

In [11]:
def summarize_access(data, group_columns):
    summary = (
        data.groupby(group_columns)
        .agg(
            settlements=("name_ru", "count"),
            over_3km=("distance_m", lambda x: (x > 3000).sum()),
            over_8km=("distance_m", lambda x: (x > 8000).sum()),
            over_13km=("distance_m", lambda x: (x > 13000).sum()),
            mean_distance=("distance_km", "mean"),
            median_distance=("distance_km", "median"),
            max_distance=("distance_km", "max"),
        )
        .reset_index()
    )

    for distance_km in (3, 8, 13):
        summary[f"pct_over_{distance_km}km"] = (
            summary[f"over_{distance_km}km"]
            / summary["settlements"]
            * 100
        ).round(1)

    distance_columns = [
        "mean_distance",
        "median_distance",
        "max_distance",
    ]

    summary[distance_columns] = (
        summary[distance_columns].round(2)
    )

    return summary


region_stats = summarize_access(
    access_clean,
    ["adm1_pcode", "region"],
)

district_stats = summarize_access(
    access_clean,
    [
        "adm1_pcode",
        "adm2_pcode",
        "region",
        "district",
    ],
)

region_stats.sort_values(
    "pct_over_8km",
    ascending=False,
)

,adm1_pcode,region,settlements,over_3km,over_8km,over_13km,mean_distance,median_distance,max_distance,pct_over_3km,pct_over_8km,pct_over_13km
6,BY007,Vitebsk,6223,3799,381,13,3.92,3.62,19.17,61.0,6.1,0.2
2,BY003,Grodno,4479,2745,208,5,3.84,3.57,14.80,61.3,4.6,0.1
1,BY002,Gomel,2302,1134,76,0,3.15,2.95,12.27,49.3,3.3,0.0
5,BY006,Mogilev,3168,1773,86,0,3.50,3.28,12.30,56.0,2.7,0.0
3,BY004,Minsk,5412,2910,127,0,3.33,3.18,12.76,53.8,2.3,0.0
0,BY001,Brest,2230,1000,46,1,2.93,2.76,13.53,44.8,2.1,0.0
4,BY005,Minsk City,2,0,0,0,1.14,1.14,1.65,0.0,0.0,0.0


In [12]:
district_stats.sort_values(
    "pct_over_13km",
    ascending=False,
).head(20)

,adm1_pcode,adm2_pcode,region,district,settlements,over_3km,over_8km,over_13km,mean_distance,median_distance,max_distance,pct_over_3km,pct_over_8km,pct_over_13km
111,BY007,BY007014,Vitebsk,Rossony,227,184,52,7,5.80,5.55,19.17,81.1,22.9,3.1
39,BY003,BY003003,Grodno,Grodno,415,310,65,5,5.04,4.77,14.80,74.7,15.7,1.2
3,BY001,BY001004,Brest,Drogichin,152,67,4,1,2.91,2.60,13.53,44.1,2.6,0.7
106,BY007,BY007009,Vitebsk,Liozno,270,195,24,2,4.69,4.54,14.15,72.2,8.9,0.7
118,BY007,BY007021,Vitebsk,Vitebsk,430,252,36,2,4.02,3.49,16.05,58.6,8.4,0.5
114,BY007,BY007017,Vitebsk,Shumilino,261,190,28,1,4.78,4.69,13.64,72.8,10.7,0.4
104,BY007,BY007007,Vitebsk,Gorodok,381,281,73,1,5.23,5.00,13.12,73.8,19.2,0.3
66,BY004,BY004013,Minsk,Nesvizh,140,45,0,0,2.34,2.39,6.07,32.1,0.0,0.0
76,BY005,BY005001,Minsk City,Minsk City,2,0,0,0,1.14,1.14,1.65,0.0,0.0,0.0
85,BY006,BY006009,Mogilev,Hotimsk,94,60,10,0,4.53,4.72,11.30,63.8,10.6,0.0


## 9. Add administrative boundary geometries

HDX administrative boundaries are joined by stable administrative codes. Boundary names are retained as the final `region` and `district` labels, while duplicate name columns are avoided.

In [13]:
regions = (
    gpd.read_file(
        "map_data/blr_admin1.geojson"
    )
    .rename(columns={"adm1_name": "region"})
)

districts = (
    gpd.read_file(
        "map_data/blr_admin2.geojson"
    )
    .rename(
        columns={
            "adm1_name": "region",
            "adm2_name": "district",
        }
    )
)

regions = regions[
    [
        "adm1_pcode",
        "region",
        "center_lat",
        "center_lon",
        "geometry",
    ]
].copy()

districts = districts[
    [
        "adm2_pcode",
        "district",
        "adm1_pcode",
        "region",
        "center_lat",
        "center_lon",
        "geometry",
    ]
].copy()

In [14]:
# Names come from the boundary files; statistics are joined by codes
district_analysis = districts.merge(
    district_stats.drop(
        columns=["region", "district"]
    ),
    on=["adm1_pcode", "adm2_pcode"],
    how="left",
)

region_analysis = regions.merge(
    region_stats.drop(columns="region"),
    on="adm1_pcode",
    how="left",
)

print("District polygons:", len(district_analysis))
print("Region polygons:", len(region_analysis))

District polygons: 119
Region polygons: 7


## 10. Load and clean Belstat population and salary data

Population by age and average monthly salary are loaded from Belstat exports.

Territory names and numeric fields are cleaned immediately. Administrative unit names are also separated from suffixes such as `district`, `region`, and `city of` before any matching takes place.

In [15]:
population_en = pd.read_csv(
    "belstat/population_belarus_2025_belstat.csv",
    sep=";",
    skiprows=2,
    names=[
        "territory",
        "age",
        "population",
    ],
)

salary_en = pd.read_csv(
    "belstat/earnings_belarus_2025_belstat.csv",
    sep=";",
    skiprows=2,
    names=[
        "territory",
        "salary",
    ],
)

population_en["territory"] = (
    population_en["territory"]
    .astype("string")
    .str.strip()
)

salary_en["territory"] = (
    salary_en["territory"]
    .astype("string")
    .str.strip()
)

population_en["population"] = pd.to_numeric(
    population_en["population"]
    .astype("string")
    .str.replace(r"\s", "", regex=True),
    errors="coerce",
)

salary_en["salary"] = pd.to_numeric(
    salary_en["salary"]
    .astype("string")
    .str.replace(",", ".", regex=False)
    .str.replace(r"\s", "", regex=True),
    errors="coerce",
)

In [16]:
# Districts, cities, and towns
population_admin_units = population_en[
    population_en["territory"].str.contains(
        r"district$|city of$|town of$",
        case=False,
        regex=True,
        na=False,
    )
].copy()

population_admin_units["district_type"] = (
    population_admin_units["territory"]
    .str.extract(
        r"(district|city of|town of)$",
        expand=False,
    )
)

population_admin_units["district"] = (
    population_admin_units["territory"]
    .str.replace(" district", "", regex=False)
    .str.replace(", city of", "", regex=False)
    .str.replace(", town of", "", regex=False)
)

# Regions
region_population = population_en[
    population_en["territory"].str.endswith(
        "region",
        na=False,
    )
].copy()

region_population["region"] = (
    region_population["territory"]
    .str.replace(" region", "", regex=False)
)

In [17]:
# Apply the same name cleaning to the salary data
salary_admin_units = salary_en[
    salary_en["territory"].str.contains(
        r"district$|city of$|town of$",
        case=False,
        regex=True,
        na=False,
    )
].copy()

salary_admin_units["district_type"] = (
    salary_admin_units["territory"]
    .str.extract(
        r"(district|city of|town of)$",
        expand=False,
    )
)

salary_admin_units["district"] = (
    salary_admin_units["territory"]
    .str.replace(" district", "", regex=False)
    .str.replace(", city of", "", regex=False)
    .str.replace(", town of", "", regex=False)
)

salary_regions = salary_en[
    salary_en["territory"].str.endswith(
        "region",
        na=False,
    )
].copy()

salary_regions["region"] = (
    salary_regions["territory"]
    .str.replace(" region", "", regex=False)
)

## 11. Calculate the population aged 65 and older

For each administrative area, the total population is taken from the `By all age` row. The 65+ population is the sum of the four oldest age groups.

In [18]:
age_65 = [
    "65 to 69 years of age",
    "70 to 74 years of age",
    "75 to 79 years of age",
    "80 and over",
]


def calculate_age_stats(data, name_column):
    total_population = (
        data.loc[
            data["age"].eq("By all age")
        ]
        .groupby(name_column)["population"]
        .sum()
        .rename("population_total")
    )

    older_population = (
        data.loc[
            data["age"].isin(age_65)
        ]
        .groupby(name_column)["population"]
        .sum()
        .rename("population_65plus")
    )

    result = pd.concat(
        [total_population, older_population],
        axis=1,
    ).reset_index()

    result["pct_65plus"] = (
        result["population_65plus"]
        / result["population_total"]
        * 100
    ).round(1)

    return result

In [19]:
district_population_stats = calculate_age_stats(
    population_admin_units[
        population_admin_units["district_type"].eq(
            "district"
        )
    ],
    "district",
)

region_population_stats = calculate_age_stats(
    region_population,
    "region",
)

# Minsk City is a city-level record, not Minsk District
minsk_city_population = calculate_age_stats(
    population_admin_units[
        population_admin_units["district_type"].eq("city of")
        & population_admin_units["district"].eq("Minsk")
    ],
    "district",
).rename(columns={"district": "region"})

minsk_city_population["region"] = "Minsk City"

region_population_stats = pd.concat(
    [
        region_population_stats,
        minsk_city_population,
    ],
    ignore_index=True,
)

## 12. Match Belstat district names

The boundary files and Belstat use different Latin transliterations. RapidFuzz finds the closest Belstat name for every district. Known mismatches are corrected manually before the merge.

The lowest-scoring matches are displayed for review.

In [20]:
population_names = (
    district_population_stats["district"]
    .dropna()
    .unique()
)

matches = []

for district_name in district_analysis["district"]:
    match, score, _ = process.extractOne(
        district_name,
        population_names,
        scorer=fuzz.ratio,
    )

    matches.append(
        {
            "district": district_name,
            "district_match": match,
            "score": score,
        }
    )

matches = pd.DataFrame(matches)

# Correct known transliteration mismatches
manual_name_corrections = {
    "Elsk": "Yelsk",
}

for boundary_name, belstat_name in manual_name_corrections.items():
    matches.loc[
        matches["district"].eq(boundary_name),
        "district_match",
    ] = belstat_name

matches.sort_values("score").head(20)

,district,district_match,score
22,Hojniki,Khoyniki,66.666667
76,Minsk City,Minsk,66.666667
20,Elsk,Yelsk,66.666667
71,St.Dorogi,Staryie Dorogi,69.565217
4,Gancevichi,Gantsevichy,76.190476
5,Ivacevichi,Ivatsevichy,76.190476
9,Ljahovichi,Lyakhovichy,76.190476
92,Krugloe,Krugloyе,80.000000
7,Kamenec,Kamenets,80.000000
10,Luninec,Luninets,80.000000


In [21]:
district_name_map = dict(
    zip(
        matches["district"],
        matches["district_match"],
    )
)

district_analysis["district_match"] = (
    district_analysis["district"]
    .map(district_name_map)
)

district_population_for_merge = (
    district_population_stats.rename(
        columns={"district": "district_match"}
    )
)

district_salary_for_merge = (
    salary_admin_units[
        salary_admin_units["district_type"].eq(
            "district"
        )
    ][["district", "salary"]]
    .rename(columns={"district": "district_match"})
)

district_analysis = district_analysis.merge(
    district_population_for_merge,
    on="district_match",
    how="left",
)

district_analysis = district_analysis.merge(
    district_salary_for_merge,
    on="district_match",
    how="left",
)

district_analysis.loc[
    district_analysis["population_total"].isna()
    | district_analysis["salary"].isna(),
    [
        "region",
        "district",
        "district_match",
        "population_total",
        "salary",
    ],
]

,region,district,district_match,population_total,salary


## 13. Add regional population and salary

Minsk City is taken from the Belstat `city of` record. This keeps it separate from Minsk District.

In [22]:
minsk_city_salary = (
    salary_admin_units[
        salary_admin_units["district_type"].eq("city of")
        & salary_admin_units["district"].eq("Minsk")
    ][["district", "salary"]]
    .rename(columns={"district": "region"})
)

minsk_city_salary["region"] = "Minsk City"

region_salary_stats = pd.concat(
    [
        salary_regions[["region", "salary"]],
        minsk_city_salary,
    ],
    ignore_index=True,
)

region_analysis = region_analysis.merge(
    region_population_stats,
    on="region",
    how="left",
)

region_analysis = region_analysis.merge(
    region_salary_stats,
    on="region",
    how="left",
)

region_analysis[
    [
        "region",
        "settlements",
        "over_8km",
        "over_13km",
        "median_distance",
        "max_distance",
        "pct_over_8km",
        "pct_over_13km",
        "population_total",
        "population_65plus",
        "pct_65plus",
        "salary",
    ]
]

,region,settlements,over_8km,over_13km,median_distance,max_distance,pct_over_8km,pct_over_13km,population_total,population_65plus,pct_65plus,salary
0,Brest,2230,46,1,2.76,13.53,2.1,0.0,1299912,231738,17.8,2360.4
1,Gomel,2302,76,0,2.95,12.27,3.3,0.0,1327973,233778,17.6,2357.5
2,Grodno,4479,208,5,3.57,14.80,4.6,0.1,984880,181656,18.4,2359.7
3,Minsk,5412,127,0,3.18,12.76,2.3,0.0,1456357,254109,17.4,2721.0
4,Minsk City,2,0,0,1.14,1.65,0.0,0.0,<NA>,<NA>,<NA>,<NA>
5,Mogilev,3168,86,0,3.28,12.30,2.7,0.0,971365,179960,18.5,2243.5
6,Vitebsk,6223,381,13,3.62,19.17,6.1,0.2,1072063,215457,20.1,2239.0


## 14. Export district and regional analysis

GeoJSON files are converted to WGS 84 for use in web maps. The CSV contains the same district-level attributes without geometry.

In [23]:
district_analysis_clean = district_analysis.copy()

district_analysis_clean.drop(
    columns="geometry"
).to_csv(
    "output_data/district_analysis_clean.csv",
    index=False,
)

district_analysis_clean.to_crs(
    "EPSG:4326"
).to_file(
    "output_data/district_analysis_clean.geojson",
    driver="GeoJSON",
    index=False,
)

region_analysis.to_crs(
    "EPSG:4326"
).to_file(
    "output_data/region_analysis.geojson",
    driver="GeoJSON",
    index=False,
)

print(
    "District rows with missing population:",
    district_analysis_clean["population_total"].isna().sum(),
)
print(
    "Region rows with missing population:",
    region_analysis["population_total"].isna().sum(),
)

District rows with missing population: 0
Region rows with missing population: 1


## 15. Estimate population outside the one-hour walking areas

`output_data/set_with_pop.geojson` is exported first and used in the separate QGIS walking-time analysis.

After QGIS classifies the settlements, export the settlements outside the one-hour service areas as:

`output_data/not_covered_points.csv`

Population is then joined by settlement ID. The calculation includes only settlements for which HOTOSM provides a population value, so it is reported as the share of the **known** population in the dataset.

In [25]:
# Use the same Belarusian settlement universe as the access analysis
settlements_with_population = settlements[
    settlements["id"].isin(access_clean["id"])
].copy()

settlements_with_population.to_file(
    "output_data/set_with_pop.geojson",
    driver="GeoJSON",
    index=False,
)

uncovered_population = pd.read_csv(
    "output_data/not_covered_points.csv"
)

population_for_merge = (
    settlements_with_population[
        [
            "id",
            "population",
            "population_num",
        ]
    ]
    .drop_duplicates(subset="id")
)

uncovered_population = uncovered_population.merge(
    population_for_merge,
    on="id",
    how="left",
)

matched_ids = uncovered_population["id"].isin(
    settlements_with_population["id"]
).sum()

print(
    "Settlement IDs matched:",
    matched_ids,
    "of",
    len(uncovered_population),
)
print(
    "Population known for all settlements:",
    settlements_with_population["population_num"].notna().sum(),
    "of",
    len(settlements_with_population),
)
print(
    "Population known for settlements beyond one hour:",
    uncovered_population["population_num"].notna().sum(),
    "of",
    len(uncovered_population),
)

Settlement IDs matched: 11551 of 11567
Population known for all settlements: 5294 of 22452
Population known for settlements beyond one hour: 1495 of 11567


In [ ]:
total_known_population = (
    settlements_with_population["population_num"].sum()
)

uncovered_known_population = (
    uncovered_population["population_num"].sum()
)

uncovered_population_pct = (
    uncovered_known_population
    / total_known_population
    * 100
)

print(
    f"Total known population: "
    f"{total_known_population:,.0f}"
)
print(
    f"Known population beyond one hour: "
    f"{uncovered_known_population:,.0f}"
)
print(
    f"Share of known population beyond one hour: "
    f"{uncovered_population_pct:.2f}%"
)

uncovered_population.to_csv(
    "output_data/not_covered_points_with_population.csv",
    index=False,
)

## 16. Check the nearest hospitals to Velikie Doltsy

This narrative-specific check considers hospitals and polyclinics only. Distances are straight-line distances in the projected metric CRS.

In [26]:
velikie_doltsy = access_clean[
    access_clean["name_ru"].eq("Великие Дольцы")
    & access_clean["region"].eq("Vitebsk")
].to_crs(metric_crs)

if len(velikie_doltsy) != 1:
    raise ValueError(
        "Expected one Velikie Doltsy settlement, "
        f"found {len(velikie_doltsy)}."
    )

hospital_candidates = healthcare_m[
    healthcare_m["facility_type"].isin(
        [
            "Больница",
            "Поликлиника",
        ]
    )
].copy()

settlement_point = velikie_doltsy.geometry.iloc[0]

hospital_candidates["distance_m"] = (
    hospital_candidates.geometry.distance(
        settlement_point
    )
)

hospital_candidates["distance_km"] = (
    hospital_candidates["distance_m"] / 1000
)

nearest_hospitals = (
    hospital_candidates.nsmallest(
        10,
        "distance_m",
    )[
        [
            "facility_title",
            "facility_type",
            "settlement",
            "address",
            "region",
            "district",
            "distance_km",
            "link",
        ]
    ]
    .copy()
)

nearest_hospitals["distance_km"] = (
    nearest_hospitals["distance_km"].round(2)
)

nearest_hospitals

,facility_title,facility_type,settlement,address,region,district,distance_km,link
2775,Великодолецкая больница сестринского ухода,Больница,Великие Дольцы,"ул. Е. Лось, 20А, 211484, д. Великие Дольцы, У...",Vitebsk,Ushat,0.30,https://healthcare.by/instinfo.php?orgnum=4365
3094,"Пышнянская больница сестринского ухода, больница",Больница,Пышно,"211199, д. Пышно, Лепельский р-н, Витебская обл.",Vitebsk,Lyepyel,11.98,https://healthcare.by/instinfo.php?orgnum=21333
3180,Ушачская ЦРБ,Больница,Ушачи,"ул. Советская, 74, 211524, г. Ушачи, Витебская...",Vitebsk,Ushat,16.50,https://healthcare.by/instinfo.php?orgnum=4358
2725,Березинская больница сестринского ухода с врач...,Больница,Березино,"ул. Заречная, 29, 211732, д. Березино, Докшицк...",Vitebsk,Dokshytsy,23.44,https://healthcare.by/instinfo.php?orgnum=4042
2954,Лепельская областная психиатрическая больница,Больница,Лепель,"ул. К. Маркса, 24, 211174, г. Лепель, Витебска...",Vitebsk,Lyepyel,25.61,https://healthcare.by/instinfo.php?orgnum=3687
3435,Стоматологическая поликлиника,Поликлиника,Лепель,"ул. Войкова, 69а, 211180, г. Лепель, Витебская...",Vitebsk,Lyepyel,25.74,https://healthcare.by/instinfo.php?orgnum=12484
2955,Лепельская ЦРБ,Больница,Лепель,"ул. Госпитальная, 2, 211174, г. Лепель, Витебс...",Vitebsk,Lyepyel,26.61,https://healthcare.by/instinfo.php?orgnum=4063
3080,Подсвильская районная больница,Больница,Подсвилье,"пер. Советский, 4, 211827, г.п. Подсвилье, Глу...",Vitebsk,Hlybokaye,32.02,https://healthcare.by/instinfo.php?orgnum=4012
2794,Вороничская больница сестринского ухода,Больница,Вороничи,"211656, д. Вороничи, Полоцкий р-н, Витебская обл.",Vitebsk,Polatsak,32.33,https://healthcare.by/instinfo.php?orgnum=4298
2780,Ветринская участковая больница,Больница,Ветрино,"ул. Озерная, 3а, 211434, г.п. Ветрино, Полоцки...",Vitebsk,Polatsak,38.72,https://healthcare.by/instinfo.php?orgnum=4295


## 17. Export medical facilities

The GeoJSON is saved in WGS 84 for Mapbox. The CSV contains the facility attributes and coordinates without a geometry column.

In [28]:
healthcare.to_crs(
    "EPSG:4326"
).to_file(
    "output_data/healthcare2.geojson",
    driver="GeoJSON",
    index=False,
)

healthcare_df.to_csv(
    "output_data/healthcare2.csv",
    index=False,
)

## 18. Prepare the 20 most remote settlements for Mapbox

For each of the 20 settlements with the greatest straight-line distance, the output contains:

- the settlement point;
- its nearest medical-facility point;
- a straight connection line between them.

The connection line shows the relationship between the two points. It is not a walking route.

In [29]:
top_20 = access_clean.nlargest(
    20,
    "distance_m",
).copy()

healthcare_top20 = healthcare.to_crs(
    top_20.crs
).copy()

facility_lookup = (
    healthcare_top20[
        [
            "facility_title",
            "address",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "address": "facility_address",
            "geometry": "facility_geometry",
        }
    )
    .drop_duplicates(
        subset=[
            "facility_title",
            "facility_address",
        ]
    )
)

top_20 = top_20.merge(
    facility_lookup,
    left_on=[
        "nearest_facility",
        "nearest_facility_address",
    ],
    right_on=[
        "facility_title",
        "facility_address",
    ],
    how="left",
)

top_20 = top_20.drop(
    columns=[
        "facility_title",
        "facility_address",
    ],
    errors="ignore",
)

top_20 = gpd.GeoDataFrame(
    top_20,
    geometry="geometry",
    crs=access_clean.crs,
)

In [30]:
def create_connection_line(row):
    settlement_point = row["geometry"]
    facility_point = row["facility_geometry"]

    if (
        settlement_point is None
        or facility_point is None
        or settlement_point.is_empty
        or facility_point.is_empty
    ):
        return None

    return LineString(
        [
            settlement_point,
            facility_point,
        ]
    )


top_20["connection_line"] = top_20.apply(
    create_connection_line,
    axis=1,
)

print(
    "Top-20 facilities matched:",
    top_20["facility_geometry"].notna().sum(),
    "of",
    len(top_20),
)

Top-20 facilities matched: 20 of 20


In [31]:
# Settlement point features
settlement_features = top_20[
    [
        "name_ru",
        "name_en",
        "name_latin",
        "region",
        "district",
        "nearest_facility",
        "nearest_facility_type",
        "nearest_facility_address",
        "distance_km",
        "geometry",
    ]
].copy()

settlement_features["feature_type"] = "settlement"

settlement_features = gpd.GeoDataFrame(
    settlement_features,
    geometry="geometry",
    crs=top_20.crs,
)

# Medical-facility point features
facility_features = top_20[
    [
        "nearest_facility",
        "nearest_facility_type",
        "nearest_facility_address",
        "facility_geometry",
    ]
].copy()

facility_features = facility_features.rename(
    columns={"facility_geometry": "geometry"}
)

facility_features["feature_type"] = "facility"

facility_features = gpd.GeoDataFrame(
    facility_features,
    geometry="geometry",
    crs=top_20.crs,
)

facility_features = facility_features.drop_duplicates(
    subset=[
        "nearest_facility",
        "nearest_facility_address",
    ]
).copy()

# Straight connection-line features
line_features = top_20[
    [
        "name_ru",
        "name_en",
        "name_latin",
        "nearest_facility",
        "nearest_facility_type",
        "distance_km",
        "connection_line",
    ]
].copy()

line_features = line_features.rename(
    columns={"connection_line": "geometry"}
)

line_features["feature_type"] = "connection"

line_features = gpd.GeoDataFrame(
    line_features,
    geometry="geometry",
    crs=top_20.crs,
)

# Combine all feature types into one GeoJSON
top_20_mapbox = gpd.GeoDataFrame(
    pd.concat(
        [
            settlement_features,
            facility_features,
            line_features,
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=top_20.crs,
)

top_20_mapbox = top_20_mapbox[
    top_20_mapbox.geometry.notna()
    & ~top_20_mapbox.geometry.is_empty
].copy()

top_20_mapbox.to_crs(
    "EPSG:4326"
).to_file(
    "output_data/top_20_mapbox.geojson",
    driver="GeoJSON",
    index=False,
)

## 19. Export remote settlements in the Vitebsk Region

This layer contains settlements in the Vitebsk Region located more than 8 km from the nearest medical facility.

In [32]:
vitebsk_over_8km = access_clean[
    access_clean["region"].eq("Vitebsk")
    & (access_clean["distance_m"] > 8000)
].copy()

vitebsk_over_8km = vitebsk_over_8km[
    [
        "name_ru",
        "name_en",
        "name_latin",
        "place",
        "region",
        "district",
        "nearest_facility",
        "nearest_facility_type",
        "nearest_facility_type_eng",
        "nearest_facility_address",
        "distance_m",
        "distance_km",
        "geometry",
    ]
].copy()

vitebsk_over_8km["distance_km"] = (
    vitebsk_over_8km["distance_km"].round(2)
)

vitebsk_over_8km.to_crs(
    "EPSG:4326"
).to_file(
    "output_data/vitebsk_settlements_over_8km.geojson",
    driver="GeoJSON",
    index=False,
)

print(
    "Vitebsk settlements saved:",
    len(vitebsk_over_8km),
)

Vitebsk settlements saved: 381


## Notes on interpretation

- Nearest-facility distances in this notebook are straight-line distances.
- The one-hour classification comes from the separate QGIS pedestrian-network analysis.
- Population results include only settlements with a population value in the HOTOSM source.
- The Top 20 connection lines are visual links, not calculated routes.